# Feature Engineering — E-commerce Return Prediction

## Objective
Prepare raw data for ML model training by:
- Removing redundant and leaky features
- Encoding categorical variables
- Handling class imbalance using SMOTE
- Splitting into train/test sets

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")
import os

In [4]:
df = pd.read_csv("../data/ecommerce_sales_data.csv")

In [3]:
df.head()

,order_id,year,month,quarter,is_weekend_order,platform,country,customer_segment,customer_age_group,customer_gender,...,return_reason,customer_satisfaction,session_duration_minutes,pages_viewed,cart_abandoned_before,wishlist_used,coupon_used,subscription_member,repeat_purchase_count,loyalty_points
0,ORD0000001,2022,4,Q2,Yes,AliExpress,Germany,New Customer,35-44,Male,...,NaN,Dissatisfied,78.7,2,Yes,No,No,No,0,0
1,ORD0000002,2023,6,Q2,Yes,Flipkart,Australia,Loyal Customer,35-44,Male,...,NaN,Neutral,18.2,7,No,Yes,No,No,9,198
2,ORD0000003,2022,2,Q1,Yes,Amazon,Turkey,VIP Customer,45-54,Male,...,NaN,Very Dissatisfied,5.1,1,No,No,No,Yes,30,1020
3,ORD0000004,2022,7,Q3,No,AliExpress,USA,New Customer,35-44,Male,...,NaN,Satisfied,22.0,6,Yes,No,No,Yes,0,0
4,ORD0000005,2024,4,Q2,No,Daraz,Australia,Loyal Customer,35-44,Female,...,NaN,Satisfied,4.5,8,Yes,No,Yes,No,13,598


In [7]:
df.shape

(10000, 38)

In [14]:
df.dtypes

order_id                        str
year                          int64
month                         int64
quarter                         str
is_weekend_order                str
platform                        str
country                         str
customer_segment                str
customer_age_group              str
customer_gender                 str
acquisition_channel             str
device_used                     str
product_category                str
unit_price_usd              float64
quantity_ordered              int64
order_value_usd             float64
discount_pct                  int64
discount_amount_usd         float64
final_amount_usd            float64
shipping_cost_usd           float64
tax_amount_usd              float64
total_paid_usd              float64
payment_method                  str
shipping_method                 str
delivery_days                 int64
product_rating                int64
review_submitted                str
returned                    

### Drop Redundant and Leaky Columns
Data Leakage Warning:** Using post-purchase columns would give
artificially high accuracy but fail completely in real deployment.

In [5]:
columns_to_drop = [
    # Identifier — just a row number, no pattern
    "order_id",
    
    # Redundant price columns — highly correlated with order_value_usd
    "final_amount_usd",
    "total_paid_usd",
    "tax_amount_usd",
    "discount_amount_usd",
    
    # Redundant loyalty column — 0.91 correlated with repeat_purchase_count
    "loyalty_points",
    
    # DATA LEAKAGE — only available AFTER return happens
    # Using these would be cheating the model
    "return_reason",
    "customer_satisfaction",
    "product_rating",      # rating given after purchase/return
    "review_submitted",    # review submitted after purchase
    
    # Time columns — too granular, quarter already covers this
    "month",
    "quarter",
]

df_clean = df.drop(columns=columns_to_drop)
print(f"After dropping: {df_clean.shape}")
print(f"Remaining columns:\n{df_clean.columns.tolist()}")

After dropping: (10000, 26)
Remaining columns:
['year', 'is_weekend_order', 'platform', 'country', 'customer_segment', 'customer_age_group', 'customer_gender', 'acquisition_channel', 'device_used', 'product_category', 'unit_price_usd', 'quantity_ordered', 'order_value_usd', 'discount_pct', 'shipping_cost_usd', 'payment_method', 'shipping_method', 'delivery_days', 'returned', 'session_duration_minutes', 'pages_viewed', 'cart_abandoned_before', 'wishlist_used', 'coupon_used', 'subscription_member', 'repeat_purchase_count']


After dropping redundant and leaky columns, we reduced from 38 → 24 features. 
All remaining features are available at the time of order placement — no data leakage.

### Encode Target Variable
Convert Yes/No → 1/0 for ML model compatibility

In [6]:
# Target variable
df_clean["returned"] = (df_clean["returned"] == "Yes").astype(int)

print("Target variable distribution:")
print(df_clean["returned"].value_counts())
print(f"\nReturn rate: {df_clean['returned'].mean()*100:.1f}%")

Target variable distribution:
returned
0    9359
1     641
Name: count, dtype: int64

Return rate: 6.4%


### Encode Categorical Columns

In [7]:
# Identify categorical columns
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}\n")

# Label Encoding — converts each category to a number
# Amazon=0, Flipkart=1, Etsy=2 etc.
le = LabelEncoder()

for col in categorical_cols:
    df_clean[col] = le.fit_transform(df_clean[col])
    print(f"Encoded: {col}")

print(f"\nFinal shape: {df_clean.shape}")
df_clean.head(3)

Categorical columns to encode: ['is_weekend_order', 'platform', 'country', 'customer_segment', 'customer_age_group', 'customer_gender', 'acquisition_channel', 'device_used', 'product_category', 'payment_method', 'shipping_method', 'cart_abandoned_before', 'wishlist_used', 'coupon_used', 'subscription_member']

Encoded: is_weekend_order
Encoded: platform
Encoded: country
Encoded: customer_segment
Encoded: customer_age_group
Encoded: customer_gender
Encoded: acquisition_channel
Encoded: device_used
Encoded: product_category
Encoded: payment_method
Encoded: shipping_method
Encoded: cart_abandoned_before
Encoded: wishlist_used
Encoded: coupon_used
Encoded: subscription_member

Final shape: (10000, 26)


,year,is_weekend_order,platform,country,customer_segment,customer_age_group,customer_gender,acquisition_channel,device_used,product_category,...,shipping_method,delivery_days,returned,session_duration_minutes,pages_viewed,cart_abandoned_before,wishlist_used,coupon_used,subscription_member,repeat_purchase_count
0,2022,1,0,5,2,2,1,3,1,7,...,4,9,0,78.7,2,1,0,0,0,0
1,2023,1,4,0,1,2,1,1,1,1,...,1,3,0,18.2,7,0,1,0,0,9
2,2022,1,1,11,4,3,1,4,1,3,...,2,7,0,5.1,1,0,0,0,1,30


### Split Features and Target

In [8]:
X = df_clean.drop(columns=["returned"])
y = df_clean["returned"]

print(f"Features (X): {X.shape}")
print(f"Target (y):   {y.shape}")
#print(f"\nFeature names:\n{X.columns.tolist()}")

Features (X): (10000, 25)
Target (y):   (10000,)


In [9]:
# Train Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # since there is class imbalance, ensures same 6.4% return rate in both sets
)

print(f"Training set:   {X_train.shape}")
print(f"Testing set:    {X_test.shape}")
print(f"\nReturn rate in train: {y_train.mean()*100:.1f}%")
print(f"Return rate in test:  {y_test.mean()*100:.1f}%")

Training set:   (8000, 25)
Testing set:    (2000, 25)

Return rate in train: 6.4%
Return rate in test:  6.4%


### Manage Class Imbalance - Apply SMOTE

In [10]:
print("Before SMOTE:")
print(f"  Not Returned: {sum(y_train==0)}")
print(f"  Returned:     {sum(y_train==1)}")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(f"  Not Returned: {sum(y_train_balanced==0)}")
print(f"  Returned:     {sum(y_train_balanced==1)}")
print(f"\nTotal training samples: {len(X_train_balanced)}")

Before SMOTE:
  Not Returned: 7487
  Returned:     513

After SMOTE:
  Not Returned: 7487
  Returned:     7487

Total training samples: 14974


### Save processed Datasets

In [11]:
os.makedirs("../data/processed", exist_ok=True)

In [12]:
# Save SMOTE balanced data (for Model 1)
X_train_balanced.to_csv("../data/processed/X_train_smote.csv", index=False)
pd.DataFrame(y_train_balanced, columns=["returned"]).to_csv(
    "../data/processed/y_train_smote.csv", index=False)

# Save original imbalanced train data (for Model 2 — Class Weights)
X_train.to_csv("../data/processed/X_train_original.csv", index=False)
pd.DataFrame(y_train, columns=["returned"]).to_csv(
    "../data/processed/y_train_original.csv", index=False)

# Save test data (same for both models)
X_test.to_csv("../data/processed/X_test.csv", index=False)
pd.DataFrame(y_test, columns=["returned"]).to_csv(
    "../data/processed/y_test.csv", index=False)

print("✅ All processed data saved!")
print(f"  X_train SMOTE:    {X_train_balanced.shape}")
print(f"  X_train Original: {X_train.shape}")
print(f"  X_test:           {X_test.shape}")

✅ All processed data saved!
  X_train SMOTE:    (14974, 25)
  X_train Original: (8000, 25)
  X_test:           (2000, 25)
